In [28]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [29]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [30]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")  # プリフィル：JSONだけ返させる
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

[
  {
    "task": "Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens only, must start and end with letter or number)"
  },
  {
    "task": "Create a JSON CloudFormation template snippet that defines an AWS IAM role with a trust policy allowing EC2 instances to assume it"
  },
  {
    "task": "Write a regular expression that matches valid AWS ARN (Amazon Resource Name) formats, such as 'arn:aws:service:region:account-id:resource'"
  }
]


In [31]:
# データセットをファイルに保存
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

print("dataset.json に保存しました")

dataset.json に保存しました


---

## 6. コア評価パイプラインの構築

3つの関数でパイプラインを構成します：
- `run_prompt` → テストケースをプロンプトとマージしてClaudeに送る
- `run_test_case` → 1件のテストケースを実行して採点する
- `run_eval` → データセット全体を処理する

In [32]:
def run_prompt(test_case):
    """テストケースをプロンプトとマージしてClaudeに送り、出力を返す"""
    prompt = f"""Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    """1件のテストケースを実行して採点する（スコアは仮で10）"""
    output = run_prompt(test_case)

    # TODO: 採点ロジックは次のSTEPで実装
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }


def run_eval(dataset):
    """データセット全体を処理して結果リストを返す"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results


print("評価パイプライン関数の定義完了")

評価パイプライン関数の定義完了


## 7. 評価パイプラインを実行する

In [33]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\n```python\nimport re\nfrom typing import Tuple\n\ndef validate_s3_bucket_name(bucket_name: str) -> Tuple[bool, str]:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules.\n    \n    AWS S3 Bucket Naming Rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters, numbers, and hyphens\n    - Must start with a lowercase letter or number\n    - Must end with a lowercase letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The S3 bucket name to validate\n        \n    Returns:\n        Tuple[bool, str]: A tuple of (is_valid, message)\n    \"\"\"\n    \n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False, \"Bucket name must be a string\"\n    \n    # Check length (3-63 characters)\n    if len(bucket_name) < 3:\n

---

## 8. モデル採点者の実装

採点役のClaudeに「強み・弱み・理由・スコア」をJSON形式で返させます。
スコアだけでなく理由も求めることで、採点が中間値（6点付近）に偏るのを防ぎます。

In [40]:
import re


def grade_by_model(test_case, output):
    """採点役のClaudeに回答を評価させてJSONで返す"""
    eval_prompt = f"""You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a JSON object with these fields:
- "strengths": An array of 1-3 key strengths (text only, no code snippets)
- "weaknesses": An array of 1-3 key areas for improvement (text only, no code snippets)
- "reasoning": A concise explanation (text only, no code snippets or special characters)
- "score": A number between 1-10

Important: Do NOT include any code, backslashes, or special characters in your response.
"""
    messages = []
    add_user_message(messages, eval_prompt)
    response = chat(messages, temperature=0.0)

    # レスポンスからJSONブロックを抽出してパース
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if json_match:
        return json.loads(json_match.group())
    raise ValueError(f"JSONが取得できませんでした: {response}")


print("grade_by_model 定義完了")

grade_by_model 定義完了


## 9. 採点ロジックを組み込んで評価パイプラインを更新する

In [41]:
from statistics import mean


def run_test_case(test_case):
    """1件のテストケースを実行して採点する"""
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }


def run_eval(dataset):
    """データセット全体を処理して平均スコアを表示する"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"平均スコア: {average_score}")
    return results


print("run_test_case / run_eval 更新完了")

run_test_case / run_eval 更新完了


## 10. 採点付きで評価パイプラインを実行する

In [42]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

# 各テストケースのスコアと理由を表示
for i, result in enumerate(results):
    print(f"\n--- テストケース {i+1} ---")
    print(f"タスク  : {result['test_case']['task'][:60]}...")
    print(f"スコア  : {result['score']}/10")
    print(f"理由    : {result['reasoning']}")

平均スコア: 5.666666666666667

--- テストケース 1 ---
タスク  : Write a Python function that validates an AWS S3 bucket name...
スコア  : 7/10
理由    : The solution demonstrates solid understanding of AWS S3 bucket naming rules and implements most of them correctly with good code organization and documentation. The validation logic is sound and handles the primary constraints effectively. However, there are minor inefficiencies in the validation checks and the test code is incomplete, which prevents full verification of the implementation. The omission of the dot restriction, while not strictly required by AWS, represents a gap in practical real-world applicability. Overall, this is a functional and well-intentioned solution that would work correctly for most use cases but has room for refinement in code efficiency and completeness.

--- テストケース 2 ---
タスク  : Create a JSON CloudFormation template snippet that defines a...
スコア  : 7/10
理由    : The solution is technically correct and well-documented, with va

---

## 11. コード採点者の実装

モデル採点に加えて、**構文の正しさ**と**形式（コードのみか）**をプログラムで検証します。

- `validate_json` / `validate_python` / `validate_regex` → 構文チェック
- `grade_syntax` → outputがコードのみかどうか + 構文チェックを組み合わせる

In [43]:
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(output, test_case):
    """フォーマット（コードのみか）と構文の正しさを採点する"""
    fmt = test_case.get("format", "")

    # マークダウンや説明文が含まれていたらフォーマット違反で0点
    if "```" in output or output.strip().startswith("#"):
        return 0

    if fmt == "json":
        return validate_json(output)
    elif fmt == "python":
        return validate_python(output)
    elif fmt == "regex":
        return validate_regex(output)

    # format未指定の場合はフォーマットチェックのみ（構文チェックなし）
    return 10


print("コード採点関数の定義完了")

コード採点関数の定義完了


## 12. データセットに format フィールドを追加して再生成する

In [44]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python"
  }
]
```

* The "format" field must be one of: "python", "json", "regex"
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects (one per format type).
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
print("\ndataset.json を更新しました")

[
  {
    "task": "Parse an AWS S3 bucket name and object key from an S3 URI string like 's3://my-bucket/path/to/object.txt' and return them as separate values",
    "format": "python"
  },
  {
    "task": "Create a JSON CloudFormation template snippet for an AWS IAM role that allows EC2 instances to assume the role",
    "format": "json"
  },
  {
    "task": "Write a regex pattern that matches valid AWS Lambda function names (must start with a letter or underscore, contain only alphanumeric characters and hyphens, and be 1-64 characters long)",
    "format": "regex"
  }
]

dataset.json を更新しました


## 13. モデルスコアとコードスコアを統合して評価パイプラインを更新する

In [45]:
def run_prompt(test_case):
    """プロンプトv1: シンプルな指示のみ（改善前のベースライン）"""
    prompt = f"""Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    # モデルスコアとコードスコアの平均
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": model_grade["reasoning"],
    }


def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"平均スコア: {average_score:.2f}")
    return results


print("評価パイプライン更新完了")

評価パイプライン更新完了


## 14. ベースライン評価を実行する（改善前のスコアを記録）

In [46]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

for i, result in enumerate(results):
    print(f"\n--- テストケース {i+1} ({result['test_case'].get('format', '?')}) ---")
    print(f"タスク        : {result['test_case']['task'][:60]}...")
    print(f"モデルスコア  : {result['model_score']}/10")
    print(f"構文スコア    : {result['syntax_score']}/10")
    print(f"合計スコア    : {result['score']}/10")
    print(f"理由          : {result['reasoning']}")

平均スコア: 3.50

--- テストケース 1 (python) ---
タスク        : Parse an AWS S3 bucket name and object key from an S3 URI st...
モデルスコア  : 7/10
構文スコア    : 0/10
合計スコア    : 3.5/10
理由          : The solution demonstrates solid fundamental parsing logic with appropriate error handling and documentation. The string manipulation approach is straightforward and efficient for the stated task. However, the validation is somewhat rigid by requiring an object key when S3 URIs can legitimately reference just a bucket, and it lacks comprehensive validation against actual AWS S3 bucket naming conventions. The incomplete test output is a minor presentation issue but raises questions about verification. Overall, this is a functional solution that handles the basic use case well but could be more flexible and robust.

--- テストケース 2 (json) ---
タスク        : Create a JSON CloudFormation template snippet for an AWS IAM...
モデルスコア  : 6/10
構文スコア    : 0/10
合計スコア    : 3.0/10
理由          : The solution demonstrates solid unde

---

## 15. プロンプトを改善してスコアを比較する（演習）

v1の問題点：コードのみを返す指示がない → マークダウンや説明文が混じる → 構文スコア0

v2の改善：
- コードのみを返すよう明示
- 説明・コメント・コードブロック禁止を追加
- プリフィルでコード生成を誘導

In [48]:
def run_prompt_v2(test_case):
    """プロンプトv2: コードのみを返すよう明示した改善版"""
    prompt = f"""Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments, commentary, or explanation
* Do not use markdown code blocks
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")  # 末尾の \n を削除
    output = chat(messages, stop_sequences=["```"])
    return output.strip()


def run_eval_compare(dataset, prompt_fn, label):
    """指定したプロンプト関数で評価して平均スコアを返す"""
    results = []
    for test_case in dataset:
        output = prompt_fn(test_case)
        model_grade = grade_by_model(test_case, output)
        model_score = model_grade["score"]
        syntax_score = grade_syntax(output, test_case)
        score = (model_score + syntax_score) / 2
        results.append({
            "test_case": test_case,
            "output": output,
            "model_score": model_score,
            "syntax_score": syntax_score,
            "score": score,
        })

    avg = mean([r["score"] for r in results])
    print(f"\n【{label}】平均スコア: {avg:.2f}")
    for i, r in enumerate(results):
        fmt = r["test_case"].get("format", "?")
        print(f"  テストケース{i+1}({fmt}): モデル={r['model_score']} 構文={r['syntax_score']} 合計={r['score']}")
    return avg


with open("dataset.json", "r") as f:
    dataset = json.load(f)

avg_v1 = run_eval_compare(dataset, lambda tc: run_prompt(tc), "v1: 改善前")
avg_v2 = run_eval_compare(dataset, run_prompt_v2, "v2: 改善後")

print(f"\nスコア差分: {avg_v2 - avg_v1:+.2f}")


【v1: 改善前】平均スコア: 3.50
  テストケース1(python): モデル=7 構文=0 合計=3.5
  テストケース2(json): モデル=6 構文=0 合計=3.0
  テストケース3(regex): モデル=8 構文=0 合計=4.0

【v2: 改善後】平均スコア: 8.00
  テストケース1(python): モデル=6 構文=10 合計=8.0
  テストケース2(json): モデル=6 構文=10 合計=8.0
  テストケース3(regex): モデル=6 構文=10 合計=8.0

スコア差分: +4.50


---

## 16. PromptEvaluator クラスの実装

今まで個別に作ってきた機能（データセット生成・実行・LLM採点）を1つのクラスにまとめます。

In [52]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_dataset(self, task_description, prompt_inputs_spec, output_file, num_cases=3):
        """テストケースを自動生成してファイルに保存する"""
        inputs_desc = "\n".join([f'- "{k}": {v}' for k, v in prompt_inputs_spec.items()])
        keys = list(prompt_inputs_spec.keys())
        example = {k: f"<{v}>" for k, v in prompt_inputs_spec.items()}

        prompt = f"""Generate {num_cases} test cases for evaluating the following task:
"{task_description}"

Each test case must be a JSON object with these fields:
{inputs_desc}

Example:
```json
[{json.dumps(example, ensure_ascii=False)}]
```

Generate {num_cases} diverse and realistic test cases as a JSON array.
"""
        messages = []
        add_user_message(messages, prompt)
        add_assistant_message(messages, "```json")
        text = chat(messages, stop_sequences=["```"])
        dataset = json.loads(text)

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)
        print(f"{len(dataset)}件のテストケースを {output_file} に保存しました")
        return dataset

    def _grade(self, task_description, prompt_inputs, output, extra_criteria=""):
        """LLM採点者がスコアと理由を返す"""
        criteria_section = f"\nAdditional criteria:\n{extra_criteria}" if extra_criteria else ""
        eval_prompt = f"""You are an expert evaluator. Score this AI-generated response.

Task: {task_description}
Input: {json.dumps(prompt_inputs, ensure_ascii=False)}
Response: {output}

Evaluate based on:
- Relevance and correctness
- Completeness{criteria_section}

Return a JSON object with:
- "score": integer 1-10
- "reasoning": one sentence explanation (no special characters or code)
"""
        messages = []
        add_user_message(messages, eval_prompt)
        response = chat(messages, temperature=0.0)
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        return {"score": 5, "reasoning": "採点結果のパースに失敗しました"}

    def run_evaluation(self, run_prompt_function, dataset_file, extra_criteria="", task_description=""):
        """データセット全体を評価して結果を返す"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        for i, prompt_inputs in enumerate(dataset):
            output = run_prompt_function(prompt_inputs)
            grade = self._grade(task_description, prompt_inputs, output, extra_criteria)
            results.append({
                "prompt_inputs": prompt_inputs,
                "output": output,
                "score": grade["score"],
                "reasoning": grade["reasoning"],
            })
            print(f"テストケース {i+1}: {grade['score']}/10 - {grade['reasoning']}")

        avg = mean([r["score"] for r in results])
        print(f"\n平均スコア: {avg:.2f}/10")
        return results


print("PromptEvaluator クラスの定義完了")

PromptEvaluator クラスの定義完了


## 17. アスリート向け食事プランのベースライン評価

In [53]:
TASK_DESCRIPTION = "アスリート1人分の1日の食事プランをコンパクトかつ簡潔に作成してください"

evaluator = PromptEvaluator(max_concurrent_tasks=3)

# テストデータを生成
dataset = evaluator.generate_dataset(
    task_description=TASK_DESCRIPTION,
    prompt_inputs_spec={
        "height": "アスリートの身長（cm）",
        "weight": "アスリートの体重（kg）",
        "goal": "アスリートの目標",
        "restrictions": "食事制限",
    },
    output_file="meal_dataset.json",
    num_cases=3,
)

print(json.dumps(dataset, indent=2, ensure_ascii=False))

3件のテストケースを meal_dataset.json に保存しました
[
  {
    "height": 180,
    "weight": 75,
    "goal": "筋肉増加",
    "restrictions": "なし"
  },
  {
    "height": 165,
    "weight": 58,
    "goal": "体脂肪率低下",
    "restrictions": "乳製品アレルギー"
  },
  {
    "height": 188,
    "weight": 85,
    "goal": "持久力向上",
    "restrictions": "ベジタリアン"
  }
]


In [54]:
# ベースライン：意図的にシンプルな指示のみのプロンプト
def run_prompt_meal_v1(prompt_inputs):
    prompt = f"""この人は何を食べるべきですか？

- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v1,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria="""
回答には以下を含めること：
- 1日の総カロリー
- 三大栄養素（タンパク質・脂質・炭水化物）の内訳
- 各食事の具体的な食品・量・タイミング
""",
)

テストケース 1: 6/10 - The response correctly calculates BMI and provides appropriate macronutrient targets for muscle gain with good food examples, but fails to meet the core requirement of providing a specific daily meal plan with concrete foods, quantities, and meal timings as explicitly requested in the task criteria.
テストケース 2: 5/10 - The response correctly identifies the user's BMI and provides relevant dietary guidance for fat loss with dairy allergy accommodation, but it fails to meet the core requirements by not including specific daily calorie targets, detailed macronutrient breakdowns with quantities, or a concrete meal-by-meal plan with timing and portion sizes.
テストケース 3: 5/10 - The response appropriately addresses the vegetarian restriction and endurance goals with relevant food sources and key nutrients, but fails to meet critical requirements by omitting specific daily calorie targets, macronutrient breakdowns with percentages, and detailed meal timing with portion sizes.

平均スコ

---

## 18. 改善1: 冒頭の一文を明確・直接的に書き直す

**v1の問題**: 「この人は何を食べるべきですか？」→ 曖昧な質問形式

**v2の改善**: 動作動詞で始め、何を・誰のために・どんな制約で作るかを明示する
- `Generate`（生成する）などの直接的な動詞から始める
- 質問ではなく指示の形にする

In [55]:
def run_prompt_meal_v2(prompt_inputs):
    prompt = f"""アスリートの食事制限に合わせた1日の食事プランを作成してください。

- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


extra_criteria = """
回答には以下を含めること：
- 1日の総カロリー
- 三大栄養素（タンパク質・脂質・炭水化物）の内訳
- 各食事の具体的な食品・量・タイミング
"""

print("=== v1: 改善前 ===")
results_v1 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v1,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v2: 冒頭を明確・直接的に改善 ===")
results_v2 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v2,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v1 = mean([r["score"] for r in results_v1])
avg_v2 = mean([r["score"] for r in results_v2])
print(f"\nスコア差分: {avg_v2 - avg_v1:+.2f} ({avg_v1:.2f} → {avg_v2:.2f})")

=== v1: 改善前 ===
テストケース 1: 6/10 - The response correctly calculates daily calorie needs and macronutrient ratios with appropriate recommendations for muscle gain, but fails to meet the core requirement of providing specific meal-by-meal breakdown with concrete food quantities and timing throughout the day.
テストケース 2: 5/10 - The response correctly identifies the user's profile and provides relevant dietary principles for fat loss with dairy allergy accommodation, but fails to meet the core requirements by omitting the specific daily calorie total, macronutrient breakdown percentages, and concrete meal-by-meal food items with quantities and timing that were explicitly requested.
テストケース 3: 5/10 - The response provides relevant nutritional guidelines and vegetarian protein sources appropriate for the athlete's endurance goals, but fails to meet the core requirements by not including a specific daily calorie total, detailed macronutrient breakdown with percentages, or a concrete meal-by-meal 

---

## 19. 改善2: 出力品質ガイドラインを追加する

冒頭の指示に加えて「含めること・形式・手順」を具体的に列挙することで、さらにスコアを上げます。

**2種類のガイドライン：**
- **出力品質ガイドライン** → 回答に含めるべき要素・形式・長さを指定
- **処理手順** → Claudeが答えにたどり着く前に考えるべきステップを指定

In [56]:
def run_prompt_meal_v3(prompt_inputs):
    prompt = f"""アスリートの食事制限に合わせた1日の食事プランを作成してください。

- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}

ガイドライン：
1. 1日の総カロリーを正確に記載すること
2. タンパク質・脂質・炭水化物の量（グラム）を明記すること
3. 朝食・昼食・夕食・間食それぞれの時間帯を指定すること
4. 食事制限に合った食品のみを使用すること
5. すべての食材の分量をグラム単位で記載すること
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


print("=== v2: 冒頭改善のみ ===")
results_v2 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v2,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v3: ガイドライン追加 ===")
results_v3 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v3,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v2 = mean([r["score"] for r in results_v2])
avg_v3 = mean([r["score"] for r in results_v3])
print(f"\nスコア差分: {avg_v3 - avg_v2:+.2f} ({avg_v2:.2f} → {avg_v3:.2f})")

=== v2: 冒頭改善のみ ===
テストケース 1: 7/10 - The response provides a well-structured, detailed meal plan with appropriate calorie targets, specific foods with quantities, and meal timings that align with muscle-building goals, but it has a critical flaw: the nutritional summary table at the end is incomplete and cuts off, failing to show the final totals for calories, macronutrients, and achievement rates that were explicitly required in the task criteria.
テストケース 2: 9/10 - The response excellently meets all requirements with a well-structured 2,100kcal meal plan including specific foods, portions, timing, complete macronutrient breakdowns, dairy-free alternatives for the allergy, and practical tips for fat loss while maintaining muscle, though minor improvements could include hydration guidelines and workout timing specifics.
テストケース 3: 9/10 - The response excellently provides a comprehensive one-day vegetarian meal plan with specific foods, portions, timing, and detailed nutritional breakdowns 

---

## 20. 改善3: XMLタグで構造化する

複数の変数が混在するプロンプトでは、XMLタグでセクションを区切るとClaudeが内容を正確に理解しやすくなります。
シンプルなプロンプトでは効果が薄いケースもありますが、複雑になるほど有効です。

In [57]:
def run_prompt_meal_v4(prompt_inputs):
    prompt = f"""<athlete_information>
- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
</athlete_information>

上記のアスリート情報をもとに、1日の食事プランを作成してください。

ガイドライン：
1. 1日の総カロリーを正確に記載すること
2. タンパク質・脂質・炭水化物の量（グラム）を明記すること
3. 朝食・昼食・夕食・間食それぞれの時間帯を指定すること
4. 食事制限に合った食品のみを使用すること
5. すべての食材の分量をグラム単位で記載すること
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, max_tokens=2000)


# chat関数にmax_tokensを渡せるよう更新
def chat(messages, system=None, temperature=1.0, stop_sequences=None, use_fast_model=False, max_tokens=1000):
    params = {
        "model": fast_model if use_fast_model else model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text


print("=== v3: ガイドライン追加 ===")
results_v3 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v3,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v4: XMLタグ追加 + max_tokens=2000 ===")
results_v4 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v4,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v3 = mean([r["score"] for r in results_v3])
avg_v4 = mean([r["score"] for r in results_v4])
print(f"\nスコア差分: {avg_v4 - avg_v3:+.2f} ({avg_v3:.2f} → {avg_v4:.2f})")

=== v3: ガイドライン追加 ===
テストケース 1: 9/10 - The response excellently provides a compact yet comprehensive 1-day meal plan with clear caloric targets (3,000kcal), detailed macronutrient breakdowns (150g protein, 375g carbs, 100g fat), specific foods with quantities and timing for multiple meals, though it appears slightly cut off at the end in the snack section.
テストケース 2: 7/10 - The response provides a well-structured meal plan with detailed nutritional calculations, appropriate calorie targets for fat loss, and consideration of the dairy allergy restriction, but it is incomplete as the afternoon snack and dinner sections are cut off mid-text, failing to deliver a complete full-day plan as requested.
テストケース 3: 9/10 - The response excellently meets all requirements with clear calorie targets (2,800kcal), detailed macronutrient breakdowns, specific foods with quantities and timing for multiple meals, proper consideration of vegetarian restrictions and endurance goals, though the response appear

---

## 21. 改善4: マルチショットプロンプト（例を示す）

高スコアだった回答を「理想的な出力例」としてプロンプトに組み込みます。
言葉で説明するより「こういう回答をして」と直接示す方がClaudeに伝わりやすいです。

In [58]:
# v4の評価結果から最高スコアの回答を例として使用する
best_result = max(results_v4, key=lambda r: r["score"])

EXAMPLE_INPUT = "\n".join([f"- {k}: {v}" for k, v in best_result["prompt_inputs"].items()])
EXAMPLE_OUTPUT = best_result["output"]

def run_prompt_meal_v5(prompt_inputs):
    prompt = f"""<athlete_information>
- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
</athlete_information>

上記のアスリート情報をもとに、1日の食事プランを作成してください。

ガイドライン：
1. 1日の総カロリーを正確に記載すること
2. タンパク質・脂質・炭水化物の量（グラム）を明記すること
3. 朝食・昼食・夕食・間食それぞれの時間帯を指定すること
4. 食事制限に合った食品のみを使用すること
5. すべての食材の分量をグラム単位で記載すること

以下は理想的な回答の例です。

<example>
<sample_input>
{EXAMPLE_INPUT}
</sample_input>
<ideal_output>
{EXAMPLE_OUTPUT}
</ideal_output>
この例は、食品の選択と量が詳細に記載されており、アスリートの目標と制限に沿った、よく構造化された回答です。
</example>
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, max_tokens=2000)


print("=== v4: XMLタグ追加 ===")
results_v4 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v4,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v5: マルチショット（例を追加） ===")
results_v5 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v5,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v4 = mean([r["score"] for r in results_v4])
avg_v5 = mean([r["score"] for r in results_v5])
print(f"\nスコア差分: {avg_v5 - avg_v4:+.2f} ({avg_v4:.2f} → {avg_v5:.2f})")

=== v4: XMLタグ追加 ===
テストケース 1: 9/10 - The response excellently provides a comprehensive 1-day meal plan with detailed nutritional breakdowns, specific food items with quantities and timing, macro nutrient targets and actual values, and practical notes, though the fat content slightly exceeds the target and could be better optimized.
テストケース 2: 9/10 - The response excellently meets all requirements with a comprehensive, well-structured meal plan including specific calorie targets (1,800 kcal), detailed macronutrient breakdowns, concrete foods with portions and timing, proper accommodation of dairy allergy, and practical adjustment guidelines, though minor improvements could be made in exercise-specific timing recommendations.
テストケース 3: 9/10 - The response excellently meets all requirements with precise calorie calculations (2,900kcal), detailed macronutrient breakdowns (144.6g protein, 405.4g carbs, 81g fat), specific meal timings with quantities, and appropriate vegetarian protein source